# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Agha314/FLyRank-Task-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
**Rule:**
We score only pages that receive real search traffic (has_real_volume = True); pages with negligible impressions are excluded from scoring entirely, since there isn't enough signal to judge their performance. Among the remaining pages, we prioritize those whose click-through rate falls below what their search position would normally predict — a larger CTR gap means the page is under-capturing clicks it should be getting. Finally, we weight this priority by search impression volume: a page with a large CTR gap and high impressions is a bigger opportunity than one with the same gap but few impressions, since fixing it affects more searchers. The final score therefore reflects both how much a page is underperforming and how many people that underperformance affects.

**Equation:**
`has_real_volume × max(ctr_gap, 0) × total_gsc_impressions_mar`


**Reason Code:**
1.   flagged -> page needs attention because its CTR is below what its position predicts
2.   insufficient_volume -> page receives fewer than 100 search impressions in March — not enough traffic to reliably judge its performance, so it's excluded from scoring.
3.   ctr_meets_expectation -> the page's actual CTR is at or above what its position tier would predict — no underperformance to flag

---

**Vedict:**
We compared impression-volume tiers (100+ impressions vs below 100 impressions). We found that 26.4% of high-demand pages (100+ impressions) are declining, versus only 0.95% of low-demand pages (below 100 impressions) — at first glance this looks like a real signal. But when we checked further, we found that 93% of the low-demand pages had zero clicks in both halves of March. Because the is_declining label is defined as second_half_clicks < first_half_clicks, when both halves are zero, this comparison mechanically evaluates to False — the page gets labeled 'not declining' simply because it never had traffic to decline from, not because it's genuinely stable. Verdict: MIXED / artifact-driven — the low decline-rate in the low-demand tier is mostly a label artifact, not a real signal, so it should not be used as evidence that low-volume pages are healthier.

In [1]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_Token')}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Pull the same per-page March aggregates you already trust from w03
page_month = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_gsc_impressions_mar,
        SUM(gsc_clicks)      AS total_gsc_clicks_mar,
        AVG(gsc_avg_position) AS avg_gsc_position_mar
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
""").df()

# Position tiers — same shape as the product's own tiering logic
import pandas as pd
bins   = [0, 3, 10, 20, float("inf")]
labels = ["1-3", "4-10", "11-20", "21+"]
page_month["position_tier"] = pd.cut(page_month["avg_gsc_position_mar"], bins=bins, labels=labels)

# Weighted CTR per tier — SUM(clicks)/SUM(impressions), NOT the mean of per-page ctr_mar
bucket_table = page_month.groupby("position_tier",observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "weighted_ctr_pct": 100 * g["total_gsc_clicks_mar"].sum() / g["total_gsc_impressions_mar"].sum()
    })
).reset_index()

print(bucket_table)
# Signal 2 — Volume (impressions) vs decline rate, tested at FlyRank's real quick-win floor (>=100 impressions)
page_month_full = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_gsc_impressions_mar,
        SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') AS first_half_clicks,
        SUM(gsc_clicks) FILTER (WHERE report_date > DATE '2026-03-15')  AS second_half_clicks,
        CASE WHEN SUM(gsc_clicks) FILTER (WHERE report_date > DATE '2026-03-15')
                  < SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15')
             THEN 1 ELSE 0 END AS is_declining
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
""").df()

page_month_full["volume_tier"] = page_month_full["total_gsc_impressions_mar"].apply(
    lambda x: "below 100 (low demand)" if x < 100 else "100+ (real demand)"
)

volume_bucket_table = page_month_full.groupby("volume_tier", observed=True).agg(
    n=("is_declining", "size"),
    decline_rate_pct=("is_declining", lambda x: 100 * x.mean())
).reset_index()

print(volume_bucket_table)

low_volume = page_month_full[page_month_full["volume_tier"] == "below 100 (low demand)"]
zero_both = ((low_volume["first_half_clicks"] == 0) & (low_volume["second_half_clicks"] == 0)).mean()
print(f"Share of low-volume pages with 0 clicks in both halves: {zero_both:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_5067/3084779960.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bucket_table = page_month.groupby("position_tier",observed=True).apply(


  position_tier        n  weighted_ctr_pct
0           1-3  16144.0          0.406253
1          4-10  81988.0          0.323895
2         11-20  32203.0          0.305241
3           21+  44969.0          0.133142


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

              volume_tier       n  decline_rate_pct
0      100+ (real demand)  101441         26.419298
1  below 100 (low demand)  229996          0.951756
Share of low-volume pages with 0 clicks in both halves: 93.0%


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:

# 1. Re-pull the honest per-page March features (no label-derived columns)
scored = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_gsc_impressions_mar,
        SUM(gsc_clicks)      AS total_gsc_clicks_mar,
        AVG(gsc_avg_position) AS avg_gsc_position_mar,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_mar
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id
""").df()

# 2. Assign each page to the same position tiers used in Section 1
bins   = [0, 3, 10, 20, float("inf")]
labels = ["1-3", "4-10", "11-20", "21+"]
scored["position_tier"] = pd.cut(scored["avg_gsc_position_mar"], bins=bins, labels=labels)

# 3. Map each tier to its expected CTR (from your Section 1 bucket_table, as a fraction not %)
expected_ctr_map = {
    "1-3": 0.00406253,
    "4-10": 0.00323895,
    "11-20": 0.00305241,
    "21+": 0.00133142
}
scored["expected_ctr"] = scored["position_tier"].map(expected_ctr_map).astype(float)


# 4. The rule, piece by piece
scored["ctr_gap"] = scored["expected_ctr"] - scored["ctr_mar"]
scored["has_real_volume"] = (scored["total_gsc_impressions_mar"] >= 100).astype(int)

scored["score"] = (
    scored["has_real_volume"]
    * scored["ctr_gap"].clip(lower=0)
    * scored["total_gsc_impressions_mar"]
)

# 5. Reason code + action label
scored["reason_code"] = "ctr_below_position_expectation"
scored["action"] = scored["score"].apply(lambda s: "review_ctr_fix" if s > 0 else "no_action")

def zero_reason(row):
    if row["score"] > 0:
        return "flagged"
    elif row["has_real_volume"] == 0:
        return "insufficient_volume"
    else:
        return "ctr_meets_expectation"

scored["diagnostic_note"] = scored.apply(zero_reason, axis=1)


# 6. Rank and write the CSV
ranked = scored.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(ranked.head(10))
print(f"\nTotal rows: {len(ranked)} | Flagged for review: {(ranked['action']=='review_ctr_fix').sum()}")

            client_hash_id           content_hash_id  \
0  client_23a62021009f63c4  content_44f34c0a90047651   
1  client_e547b89c05043229  content_8d7d99f109e19aa2   
2  client_73cda7b4e4f265ea  content_8e1334d6356668e3   
3  client_62f4a7e64f5e0096  content_34a70fea29d15f24   
4  client_73cda7b4e4f265ea  content_fec55986a1868d62   
5  client_62f4a7e64f5e0096  content_7c6373141eae744a   
6  client_62f4a7e64f5e0096  content_f6116743b00afc2d   
7  client_e547b89c05043229  content_306bc78dff1eb683   
8  client_62f4a7e64f5e0096  content_acbcc847f8996314   
9  client_9958f0a7ae1df715  content_cd3d932d4e1c8db0   

   total_gsc_impressions_mar  total_gsc_clicks_mar  avg_gsc_position_mar  \
0                   212404.0                  24.0              7.346909   
1                   203497.0                 289.0              2.563756   
2                   134984.0                   1.0              4.545582   
3                   143019.0                  43.0              3.219473   
4  

## 3. Top-20 review


Row 0 : Action = review_ctr_fix. Position tier 4–10 (avg 7.35), CTR 0.011% vs expected 0.32% — large gap on 212,404 impressions (only 24 clicks). What would make it wrong: a very recent title/meta change not yet reflected in how GSC reports the snippet.

Row 1 : Action = review_ctr_fix. Position tier 1–3 (avg 2.57) — top of page, yet CTR only 0.14% vs expected 0.41% (289 clicks / 283,497 impressions). What would make it wrong: this looks like the "position paradox" — a top-ranking informational page where a featured snippet or direct answer satisfies the searcher without a click; low CTR here may not mean a broken listing.

Row 2 : Action = review_ctr_fix. Position tier 4–10 (avg 4.55), but only 1 click across 134,984 impressions. What would make it wrong: this is more likely a data/tracking anomaly than a real CTR problem — worth checking days_gsc_available for this page before trusting it.

Row 3 : Action = review_ctr_fix. Position tier 4–10 (avg 3.22), CTR 0.03% vs expected 0.32% (43 clicks / 143,019 impressions). What would make it wrong: if this content is very new, position may still be settling and CTR hasn't stabilized yet.

Row 4 (content_fec5598...): Action = review_ctr_fix. Position tier 4–10 (avg 9.39), only 1 click across 124,075 impressions — same red flag as Row 2. What would make it wrong: likely a measurement/tracking gap rather than a genuine listing problem; verify before acting.

Row 5 : Action = review_ctr_fix. Position tier 4–10 (avg 5.79), CTR 0.06% vs expected 0.32% (83 clicks / 132,593 impressions). What would make it wrong: seasonal or one-off demand spike inflating impressions without matching interest in the click.

Row 6 (content_f611674...): Action = review_ctr_fix. Position tier 4–10 (avg 9.54), CTR 0.014% vs expected 0.32% (15 clicks / 107,584 impressions). What would make it wrong: page near the tier boundary (close to position 10) — a small ranking shift could move it to the 11–20 tier with a lower baseline, changing the "gap" entirely.

Row 7 : Action = review_ctr_fix. Position tier 1–3 (avg 1.49) — essentially #1, yet CTR only 0.043% vs expected 0.41% (35 clicks / 80,821 impressions). What would make it wrong: same position-paradox risk as Row 1 — a #1 informational result can legitimately have low CTR if it fully answers the query on the results page itself.

Row 8 : Action = review_ctr_fix. Position tier 4–10 (avg 3.36), CTR 0.15% vs expected 0.32% (262 clicks / 170,808 impressions) — score ≈291 (computed from the formula; screenshot cut off before this row's score printed — worth confirming against your actual output). What would make it wrong: this page has the healthiest click volume of the flagged set — closer scrutiny might show it's borderline, not a priority fix.

Row 9 : Action = review_ctr_fix. Position tier 4–10 (avg 7.79), only 4 clicks across 89,332 impressions — score ≈285 (computed, same caveat as Row 8). What would make it wrong: another very-low-click case — check for tracking gaps before treating this as a genuine CTR problem.

In [ ]:
review_cols = [
    "content_hash_id", "avg_gsc_position_mar", "position_tier",
    "total_gsc_impressions_mar", "ctr_mar", "expected_ctr", "ctr_gap", "score"
]

top10 = ranked.head(10)[review_cols]
print(top10.to_string(index=True))

            content_hash_id  avg_gsc_position_mar position_tier  total_gsc_impressions_mar   ctr_mar  expected_ctr   ctr_gap       score
0  content_44f34c0a90047651              7.346909          4-10                   212404.0  0.000113      0.003239  0.003126  663.965936
1  content_8d7d99f109e19aa2              2.563756           1-3                   203497.0  0.001420      0.004063  0.002642  537.712667
2  content_8e1334d6356668e3              4.545582          4-10                   134984.0  0.000007      0.003239  0.003232  436.206427
3  content_34a70fea29d15f24              3.219473          4-10                   143019.0  0.000301      0.003239  0.002938  420.231390
4  content_fec55986a1868d62              9.385150          4-10                   124075.0  0.000008      0.003239  0.003231  400.872721
5  content_7c6373141eae744a              5.789019          4-10                   132593.0  0.000626      0.003239  0.002613  346.462097
6  content_f6116743b00afc2d              

## 4. Weak picks + leakage check

Weak picks: Two patterns in the top-10 lower my confidence on specific rows. (1) Rows with near-zero clicks despite six-figure impressions (rows 2, 4, 9 — 1, 1, and 4 clicks respectively) look more like a data/tracking anomaly than a genuine CTR problem; before treating these as fix candidates I'd check days_gsc_available for tracking gaps. (2) Rows already ranking near position #1 (rows 1, 7 — avg positions 2.57 and 1.49) show low CTR that may reflect a legitimate "position paradox" — a top result that fully answers the query on the results page (e.g., via a featured snippet) can have low CTR without any listing problem. Both patterns are honest weaknesses of a rule this simple; a more refined version would separate "low click volume" from "low CTR relative to volume" as different reason codes.

Leakage check: Confirmed programmatically — the feature set used in scoring (total_gsc_impressions_mar, total_gsc_clicks_mar, avg_gsc_position_mar, ctr_mar, expected_ctr, ctr_gap) contains none of is_declining, second_half_clicks, first_half_clicks, or any FlyRank product-flag columns (health_score, priority_score, action_type, needs_ctr_fix, is_quick_win, refresh_tier). All features are computed from March 2026 data only — no forward-looking window was used.

In [ ]:

leakage_columns = [
    "is_declining", "second_half_clicks", "first_half_clicks",
    "health_score", "priority_score", "action_type",
    "needs_ctr_fix", "is_quick_win", "refresh_tier"
]

feature_columns_used = ["total_gsc_impressions_mar", "total_gsc_clicks_mar",
                         "avg_gsc_position_mar", "ctr_mar", "expected_ctr", "ctr_gap"]

found_leakage = [col for col in leakage_columns if col in ranked.columns and col in feature_columns_used]

print("Leakage check:")
print(f"Columns actually used in score calculation: {feature_columns_used}")
print(f"Any leakage columns found among them? {found_leakage if found_leakage else 'None — clean.'}")
print(f"\nTime window used: single month (March 2026) only — no future-window data pulled.")

Leakage check:
Columns actually used in score calculation: ['total_gsc_impressions_mar', 'total_gsc_clicks_mar', 'avg_gsc_position_mar', 'ctr_mar', 'expected_ctr', 'ctr_gap']
Any leakage columns found among them? None — clean.

Time window used: single month (March 2026) only — no future-window data pulled.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.